# Classiq Overview Tutorial in native FlagQuantum

Complete local computational Notebook workflow: native arithmetic program, synthesis, text circuit drawing and seeded sampling from executed named probabilities. Text drawing and local sampling replace cloud services; no hardware measurement is claimed.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
from collections import Counter
from random import Random
import flagquantum as fq
from flagquantum import synth
from flagquantum.drawer import draw
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc

## main

In [ ]:
@qfunc
def main(x: Output[QNum], y: Output[QNum]) -> None:
    allocate(3, x)
    hadamard_transform(x)
    y |= x**2 + 1

## run example

In [ ]:
def run_example() -> None:
    program = synth.synthesize(main, memory_limit_bytes=512 * 1024**2)
    print(draw(program.circuit))
    result = fq.run(program)
    probabilities = dict(program.probabilities(result, registers=("x", "y")))
    print("Probabilities:", probabilities)
    samples = Random(17).choices(
        list(probabilities), weights=list(probabilities.values()), k=1000
    )
    print("Counts:", dict(Counter(samples)))

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states
from examples.synthesis.tutorials.basics.quantum_programming.classiq_overview_tutorial.comparisons.circuit import (
    reference_circuit,
)

compare_states(
    synth.synthesize(main, memory_limit_bytes=512 * 1024**2),
    reference_circuit(),
    ("x", "y"),
)
print("Independent full-state comparison: passed")
